In [ ]:
# Notebook dirs
save_dir = None
docs_dir = 'docs'

# Model parameters
max_new_tokens = 4096
temperature = 0.1
context_window = 8096

# RAG index
chunk_size=512
chunk_overlap=20
top_k_documents = 3

# Ollama settings
port = None
request_timeout = 60

In [ ]:
import logging

# Disabilita i log di httpx (molto comune con client API moderni)
logging.getLogger("httpx").setLevel(logging.WARNING)

# Se usi la libreria requests/urllib3
logging.getLogger("urllib3").setLevel(logging.WARNING)

## Part 4 - RAG pipeline (2 bonus points)

In the previous part, we successfully fine-tuned our "Smart Doctor" to specialize its medical knowledge using LoRA. However, even the most advanced fine-tuned models face two major hurdles: hallucinations and outdated information. In the fast-moving medical field, a model's internal weights might not contain the very latest research papers or clinical guidelines published this morning.

To solve this, we will implement a RAG pipeline. Instead of relying solely on what the model "remembers," we will give it a database of research papers. When asked a question, the system will first search this library for relevant snippets and then present those snippets to the LLM as context. This transforms our "Smart Doctor" from a student relying on his/her memory into a specialist consulting the latest advancements in real-time.

---
**What we are going to do**

Here is an overview of what we will be going through in this assignment:

1.   Load documents
2.   Create embedding vectors for documents
3.   Create vector index for rag query
4.   Use Ollama to set up an LLM endpoint
5.   Use your created vector index and the Ollama endpoint to query your rag pipeline
6. Compare direct LLM and RAG generated responses


**Your tasks:**
* Optimize the chunk size and the chunk overlap parameters for the RAG pipeline
* Implement the main functions for the RAG pipeline
* Compare the quality of your RAG pipeline with simple LLM prompting (without RAG)


---
**How to get the documents**

The documents are already available as `docs.zip` on canvas. You can either unzip and load them in a Google Colab session, or upload them to your personal Google drive folder and use from there.This notebook will assume that you have an **unzipped** folder of documents named `docs`.

---
**Clarification**
- Reading the research papers for the assignment is not necessary. If you are curious, feel free to read them but we will not be evaluating for that.

- For comparison, we have created a small list of question-answer pairs. You can experiment with your own documents and questions as well!


---
**Grading**

You can gain up to 2 bonus points if your code is correct and you give a clear and detailed explanation of your design process and design decisions.


Let's begin!

### RAG system - A recap

There are three phases in a RAG system - **retrieval**, **augment**, **generation**. The retrieval phase can also be divided into a first phase called the pre-processing (or **indexing**) phase and the retrieval phase. In the indexing phase, you provide your documents to the pipeline, which then reads and stores them in some data format. In practice, we use a very specific type of database for this, called a vector database (hence the term indexing). In the remaining phases, a rag pipeline will fetch information from the vector database, feed them to an LLM and generate answers. Let's look at a detailed breakdown of each phase.

---

### Indexing Phase:
Before the system can answer questions, it needs to "read" the documents.

- **Knowledge base:** You start with your files (PDFs, txt, etc.).

- **Document splitting & tokenisation:** The system reads the files and split the content of each document into smaller chunks.

- **Embedding model:** An embedding model, e.g. BERT, encodes each chunk into an `embedding vector`. This vector represents the meaning of the text.

- **Vector database:** These vectors are stored in a special database designed for searching by meaning.

---

### Retrieval, Augment and Generation Phase:
In this phase, you ask questions, or, query the system and the system follows through in the following process:

- **User query:** You ask a question.

- **Embedding model:** The same embedding model from before converts your question into a query vector.

- **Retrieval:** You compare your query vector to all the stored document vectors (in the vector database) to find the most similar ones (the "nearest neighbours"). The metric to find the nearest neighbour varies, for instance you can use *cosine similarity* or *euclidean distance*.

- **Augment prompt with context:** The system takes the original question and combines it with the relevant text chunks it just found. This creates a new, more informative prompt.

- **LLM (generation):** This enriched prompt is sent to an LLM, e.g. Gemma. The LLM uses the provided context to generate an accurate and factual answer.

- **Generated answer:** The final answer is presented to you.

---
**Hints:** Review Lecture 10 before starting this part of the assignment.

### Setting up a new environment (Optional)

If you have limited GPU memory (e.g., local GPU or Google Colab T4 GPU) we recommend reinitializing the runtime and only load the necessary libraries and models. In Google Colab you can do that by going in Runtime > Restart session. Once you have done this you are ready to install the required packages, models, and datasets for this assignment part.

**Rag related packages**

We are going to use `llama-index` for building the pipeline and `ollama` for loading and inference with the selected model locally. There are other libraries such as haystack, langchain which are also popular and widely used for RAG. If you are interested, feel free to explore in your free time.

In [ ]:
import importlib, os, sys, shutil

if 'google.colab' in sys.modules:
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')

    docs_dir = '/content/drive/MyDrive/docs'

    if importlib.util.find_spec("llama_index") is None:
        !pip install -q tabulate llama-index llama-index-embeddings-huggingface llama-index-readers-file llama-index-llms-ollama sentence-transformers

    if not shutil.which('zstd'):
        !apt update -qq
        !apt install zstd -qqy

In [ ]:
# Imports
import os
import subprocess
import socket
import time
from dataclasses import dataclass
import torch

from llama_index.core import (
    VectorStoreIndex, # creates your vector database
    SimpleDirectoryReader, # reads a directory full of docs
    Settings, # let's you set your embedding and generation model on the fly
    StorageContext, # this defines how your vector database will store the vectors
    Document, # as the name suggests, a document (text and related metadata)
    Response, # for getting a response during the generation phase
    load_index_from_storage, # reloading a vector db from storage (yes you can share them!)
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from sentence_transformers import SentenceTransformer, util

**Ollama setup**

In [ ]:
local_bin = os.path.abspath(".ollama/bin/ollama")
ollama_path = shutil.which("ollama") or local_bin

if not os.path.exists(ollama_path):
        print("Download Ollama...")
        os.makedirs(".ollama", exist_ok=True)
        !wget -q -O - https://ollama.com/download/ollama-linux-amd64.tar.zst | tar --zstd -xf - -C .ollama

In [ ]:
if port is None:
    s = socket.socket()
    s.bind(("", 0))
    port = s.getsockname()[1]
    s.close()
    print("Using port ", port)

    os.makedirs("out", exist_ok=True)

    with open(f"out/ollama_{port}.out", "w") as out_file, \
        open(f"out/ollama_{port}.err", "w") as err_file:
        
        subprocess.Popen(
            [ollama_path, "serve"],
            stdout=out_file,
            stderr=err_file,
            env=dict(os.environ, OLLAMA_HOST=f"0.0.0.0:{port}")
        )

    time.sleep(10)
else:
    print(f"Ollama is already running on port {port}")

OLLAMA_HOST=f"0.0.0.0:{port}"
!OLLAMA_HOST={OLLAMA_HOST} {ollama_path} pull gemma3:270m

Using port  35897



**Deep Dive: Under the Hood of LlamaIndex**

For our RAG pipeline, we need a module which will read files from our docs directory. For this, we use `SimpleDirectoryReader`. It will read an entire directory with our documents (pdf, txt, etc.) and return a list of `Document` objects. A Document in llama-index is different than what we mean as documents. If you recall, the documents will be split into chunks. Each chunk becomes a Document object with properties which llama-index can use during all the phases of the pipeline. To create the chunks, we use `SentenceSplitter`. There are other methods as well but this one is the simplest. Feel free to explore the other methods if you are interested.

Now for the vector database. The database needs to be stored on disk first. For this it needs a `StorageContext`. Once a storage context exists, llama-index can give you access to the vector database as an `index` via `VectorStoreIndex`. So in a nutshell, the indexing phase creates an index of vectors from your documents.

---

For embedding models, we will be using the BAAI BGE Model, from Hugging Face. As for generation, we will use Gemma3-270M via Ollama as Ollama makes the process simpler.

### Section 4.1 - Building the RAG pipeline

In this section you will implement a complete RAG pipeline. We will start by defining a model configuration and some settings for `llama-index`. Then we will implement the following parts:

* Data loading: Load the unzipped research papers using `SimpleDirectoryReader`.
* Data ingestion: It consists of three parts: define the text splitter method (using `SentenceSplitter`), create a vector index (using `VectorStoreIndex` and the splitter defined above), and save the index to disk.
* Define a query engine (e.g., top_k sample similarity) and generate responses. It will use Gemma3-270M which was defined in the Llama-Index settings as our generation model.

**Critical thinking:** the following questions are designed to deepen your understanding of the RAG architecture and to drive your thinking. Feel free to have a look at them, but don't feel obliged to answer!
* Q1: We typically use encoder-only models (like BERT) for embeddings, while most LLMs are decoder-only nowadays. Since a decoder-only model (like Gemma) also creates internal representations of text, why don't we simply use the decoder's hidden states as our embeddings?
* Q2: What if the generated text in response to the augmented prompt is empty? Why might a RAG pipeline fail to generate text even if the retrieval step was successful? What fallback strategies would you implement for a "Smart Doctor" app in this scenario?

In [ ]:
# Model configuration
@dataclass
class ModelConfig:
    generation_model: str = "gemma3:270m"
    embedding_model: str = "BAAI/bge-small-en-v1.5"
    # Generation parameters
    # TIP: feel free to explore on what these generation params do
    max_new_tokens: int = max_new_tokens
    temperature: float = temperature
    context_window: int = context_window
    # Ollama settings
    ollama_base_url: str = f"http://localhost:{port}"
    request_timeout: float = request_timeout

model_config = ModelConfig()

In [ ]:
# NOTE: rerun this cell if you change the imports at the top :)

# LLM settings for llama-index
Settings.llm = Ollama(
    model=model_config.generation_model,
    base_url=model_config.ollama_base_url,
    request_timeout=model_config.request_timeout,
    temperature=model_config.temperature,
    context_window=model_config.context_window,
    num_predict=model_config.max_new_tokens,
)

Settings.embed_model = HuggingFaceEmbedding(
    model_name=model_config.embedding_model,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

In [ ]:
@dataclass
class RAGConfig:
    # Text ingestion parameters
    chunk_size: int
    chunk_overlap: int

# We will modify the chunk_size and the chunk_overlap parameters in the next section
rag_config = RAGConfig(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

In [ ]:
### YOUR IMPLEMENTATION HERE (Load documents)

def load_documents(documents_dir: str) -> list[Document]:
    """Load all supported documents from the specified directory."""
    documents = SimpleDirectoryReader(
        input_dir=documents_dir,
        recursive=True
    ).load_data()

    print(f"Loaded {len(documents)} documents from: {documents_dir}")
    return documents

documents = load_documents(docs_dir)

Loaded 145 documents from: /content/drive/MyDrive/docs


In [ ]:
### YOUR IMPLEMENTATION HERE (Ingest documents)
# use the documents you've loaded above and build a vector index
# you've to use the rag config values for chunk size and chunk overlap here
# the save_index param lets you save your index to disk
# it is recommended to save it so that you can reuse it later

def ingest_documents(documents: list[Document], rag_config: RAGConfig, persist_dir):
    transformations = [
        SentenceSplitter(chunk_size=rag_config.chunk_size, chunk_overlap=rag_config.chunk_overlap)
    ]

    vector_index = VectorStoreIndex.from_documents(
        documents,
        transformations=transformations
    )

    if persist_dir:
        vector_index.storage_context.persist(persist_dir=persist_dir)
        print(f"Index saved to: {persist_dir}")

    return vector_index

t0 = time.time()
vector_index = ingest_documents(documents, rag_config, save_dir)
setup_t = time.time() - t0

In [ ]:
### YOUR IMPLEMENTATION HERE (Define query engine)
# return a query engine from the vector_index with top_k value

def get_query_engine(vector_index: VectorStoreIndex, top_k: int):
    """
    Restituisce un query engine configurato per recuperare i top_k documenti.
    """
    # as_query_engine trasforma l'indice in un'interfaccia interrogabile
    query_engine = vector_index.as_query_engine(
        similarity_top_k=top_k
    )

    return query_engine

query_engine = get_query_engine(vector_index, top_k=top_k_documents)


In [ ]:
# This is an helper function to test the RAG pipeline
# Do not modify it, unless necessary
def query_with_rag(query: str, query_engine=query_engine) -> Response:
    assert query_engine is not None, "Query engine not initialized"

    response = query_engine.query(query)
    return response


# Run with an example query. Feel free to explore all the properties/attributes in the resp variable
query = "How many women are diagnosed with invasive breast cancer each year in the Netherlands?"
print(f"User query: {query}\n")

resp = query_with_rag(query)
print(f"Response: {resp.response}")

User query: How many women are diagnosed with invasive breast cancer each year in the Netherlands?

Response: The Netherlands has a total of 69,570 women diagnosed with invasive breast cancer each year.



Congratulations, if you have written correct code by now you should have received some response from your query, in the `resp` object. Check the attributes of the response you got. There are a few important things here:

1. Source nodes - chunks which were retrieved and are related to your query
2. Score for nodes - shows how related the retrived nodes are to your query.

You can display the source nodes and the related score with the following function.

In [ ]:
# This is an helper function to display the response with some metadata
# Do not modify it, unless necessary
def display_response(response: Response):
    """
    Takes a RAG generated response and shows the response text, source nodes and scores
    """

    # IMPLEMENT
    print("=" * 50)
    print("Generated response: ")
    print(str(response))
    print("=" * 50)

    print()

    print("=" * 50)
    print("Source nodes: ")
    for source in response.source_nodes:
        print(f"Score: {source.score}")
        print(f"Metadata: {source.metadata}")
        print("Text:")
        print(source.node.get_text())
        print()
    print("=" * 50)
    print()

display_response(resp)

Generated response: 
The Netherlands has a total of 69,570 women diagnosed with invasive breast cancer each year.


Source nodes: 
Score: 0.8471624994755435
Metadata: {'page_label': '1', 'file_name': 'ned-breast-cancer.pdf', 'file_path': '/content/drive/MyDrive/docs/ned-breast-cancer.pdf', 'file_type': 'application/pdf', 'file_size': 111503, 'creation_date': '2026-03-29', 'last_modified_date': '2026-03-21'}
Text:
RIVM Committed to health and sustainability
Breast cancer in the Netherlands
National Institute for Public Healthand the EnvironmentMinistry of Health, Welfare and Sport
Modification date 20-08-2025 | 17:34
Approximately 14,000 women per year are diagnosed with invasive breast cancer and approximately 2,400 with in-
situ breast cancer. The average age at the time of diagnosis is approximately 61 years.
Each year around 3,000 women die as a consequence of breast cancer. Approximately 1 in 8 women in the Netherlands will develop breast cancer at
some point in their lives. This m

### Section 4.2 — Optimizing retrieval
A RAG pipeline is only as good as the context it retrieves. If your chunks are too small, the model loses the broader clinical context; if they are too large, the "noise" from irrelevant sentences may confuse the LLM.

**Your Tasks:**
In this section, you will experiment with the `SentenceSplitter` parameters:
* Chunk size: modify the number of tokens per chunk.
* Chunk overlap: adjust the overlap between consecutive chunks to ensure that key medical terms or definitions aren't cut in half.

**Critical thinking:** Observe the "Source Nodes" retrieved for a specific question. Does a larger overlap lead to more coherent or repetitive answers? What is the best set of hyperparameters you found? How did you compare the different solutions?

**Hint:** also consider the time for the indexing and generation phase.

**Hint:** You can use the `test_questions` in the next part to verify the output of your model with different sets of hyperparameters.

In [ ]:
print(f"{chunk_size=}")
print(f"{chunk_overlap=}")

chunk_size=512
chunk_overlap=20


### Section 4.3 — Compare RAG vs. without RAG
To conclude the assignment, you will compare the RAG system you just implemented with the model generation without RAG.

**Your Tasks:**
* Implement a function to ask the LLM a highly specific technical question from one of the research papers without using the RAG pipeline and then ask the same question using your pipeline.

* Compare the results in terms of generation quality, answer format, hallucinations, etc.

For the final comparison you can use the list of questions we provided but you can also create your own list.



In [ ]:
# This is an helper function to query an LLM without using the RAG pipleine
# Do not modify it, unless necessary
def query_without_rag(query: str, llm=Settings.llm):
    response = llm.complete(query)
    return response.text

query = "How many women are diagnosed with invasive breast cancer each year in the Netherlands?"
print(f"User query: {query}\n")

resp_without_rag = query_without_rag(query)
print(f"Response: {resp_without_rag}")

User query: How many women are diagnosed with invasive breast cancer each year in the Netherlands?

Response: According to the Dutch Cancer Registry, there are approximately **1,500 women diagnosed with invasive breast cancer each year**.



In [ ]:
test_questions = [
    {
        "question": "How many women are diagnosed with invasive breast cancer each year in the Netherlands?",
        "answer": "Approximately 14,000 women per year are diagnosed with invasive breast cancer in the Netherlands."
    },
    {
        "question": "What is the average age of a woman at the time of a breast cancer diagnosis in the Netherlands?",
        "answer": "The average age at the time of diagnosis is approximately 61 years."
    },
    {
        "question": "What is the lifetime risk of a woman developing breast cancer in the Netherlands?",
        "answer": "Approximately 1 in 8 women in the Netherlands will develop breast cancer at some point in their lives."
    },
    {
        "question": "How does the annual cancer death rate in the Netherlands compare to the broader European average?",
        "answer": "The annual death rate for cancer patients in the Netherlands is higher than the European average, recording 267 deaths for every 100,000 people compared to the European average of 247."
    },
    {
        "question": "What are the most common types of cancer found in the Netherlands?",
        "answer": "The most common cancers in the Netherlands are prostate, breast, colorectal (colon), and lung cancer."
    },
    {
        "question": "How many people worldwide died from cancer in the year 2020?",
        "answer": "Cancer accounted for nearly 10 million deaths worldwide in 2020, representing nearly one in six of all deaths globally."
    },
    {
        "question": "What are the top five most common causes of cancer death globally?",
        "answer": "The most common causes of cancer death worldwide are lung, colon and rectum, liver, stomach, and breast cancers."
    },
    {
        "question": "What proportion of cancer deaths are attributed to common lifestyle risk factors?",
        "answer": "Around one-third of deaths from cancer are due to tobacco use, high body mass index, alcohol consumption, low fruit and vegetable intake, and a lack of physical activity."
    },
    {
        "question": "What is the medical definition of metastasis?",
        "answer": "Metastasis is the process where abnormal cancer cells rapidly grow beyond their usual boundaries, invade adjoining parts of the body, and spread to other organs."
    },
    {
        "question": "Can viral infections cause cancer?",
        "answer": "Yes, cancer-causing infections such as human papillomavirus (HPV) and hepatitis are responsible for approximately 30% of cancer cases in low- and lower-middle-income countries."
    },
    {
        "question": "Approximately how many children develop cancer each year?",
        "answer": "Each year, approximately 400,000 children develop cancer globally."
    },
    {
        "question": "What does the American Cancer Society's acronym 'C-A-U-T-I-O-N' stand for regarding early cancer warning signs?",
        "answer": "It stands for: Change in bowel or bladder habits, A sore that does not heal, unusual bleeding or discharge, Thickening or lump in the breast or elsewhere, Indigestion or difficulty in swallowing, Obvious change in a wart or mole, and Nagging cough or hoarseness."
    },
    {
        "question": "According to a study on cancer awareness, what was the most widely recognized warning symptom of cancer among the public?",
        "answer": "The most frequently recognized cancer symptom was an 'unexplained lump or swelling', which was identified by 72.8% of the participants."
    },
    {
        "question": "What are common emotional barriers that prevent individuals from seeking early medical advice for potential cancer symptoms?",
        "answer": "Common emotional barriers include being too scared (fear) and worrying about what the doctor might find during the diagnosis."
    },
    {
        "question": "What are some of the acute, reversible side effects commonly associated with chemotherapy?",
        "answer": "Acute reversible side effects of chemotherapy include alopecia (hair loss), nausea, vomiting, fatigue, and myelosuppression."
    }
]

In [ ]:
eval_model = SentenceTransformer('all-MiniLM-L6-v2')

def get_score(a: str, b: str) -> float:
    emb = eval_model.encode([a, b], convert_to_tensor=True, show_progress_bar=False)
    return util.pytorch_cos_sim(emb[0], emb[1]).item()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
### YOUR IMPLEMENTATION HERE (Comparison with and without RAG)
times_no_rag = []
times_rag = []
scores_no_rag = []
scores_rag = []
node_scores_rag = []

for i, entry in enumerate(test_questions, 1):
    question, gold = entry["question"], entry["answer"]

    # without RAG
    t_start = time.time()
    no_rag = query_without_rag(question)
    dt_plain = time.time() - t_start

    score_plain = get_score(no_rag, gold)

    times_no_rag.append(dt_plain)
    scores_no_rag.append(score_plain)

    # with RAG
    t_start = time.time()
    with_rag = query_with_rag(question)
    dt_rag = time.time() - t_start

    score_rag = get_score(with_rag.response, gold)
    avg_n_score = sum(n.score for n in with_rag.source_nodes) / len(with_rag.source_nodes)

    times_rag.append(dt_rag)
    scores_rag.append(score_rag)
    node_scores_rag.append(avg_n_score)

    # output
    print(f"Q{i}: {question.strip()}")
    print(f"┣ EXPECTED")
    print(f"┃   {gold.strip()}")
    print(f"┣ PLAIN:    ({score_plain:.3f} | {dt_plain:.2f}s)")
    print(f"┃   {no_rag.strip()}")
    print(f"┗ WITH RAG: ({score_rag:.3f} | {dt_rag:.2f}s | Node Score: {avg_n_score:.3f})")
    print(f"    {with_rag.response.strip()}")
    print("-" * 50)

Q1: How many women are diagnosed with invasive breast cancer each year in the Netherlands?
┃
┣ EXPECTED
┃   Approximately 14,000 women per year are diagnosed with invasive breast cancer in the Netherlands.
┣ PLAIN:    (0.944 | 0.40s)
┃   The number of women diagnosed with invasive breast cancer each year in the Netherlands is approximately **1.5 million**.
┗ WITH RAG: (0.939 | 9.33s | Node Score: 0.807)
    The Netherlands has a high prevalence of invasive breast cancer, with approximately 14,000 women diagnosed with invasive breast cancer each year. This is a significant proportion of the total number of women in the Netherlands, with a ten-year prevalence of 128,000.
--------------------------------------------------
Q2: What is the average age of a woman at the time of a breast cancer diagnosis in the Netherlands?
┃
┣ EXPECTED
┃   The average age at the time of diagnosis is approximately 61 years.
┣ PLAIN:    (0.577 | 0.66s)
┃   The average age of a woman at the time of a breast can

In [ ]:
from tabulate import tabulate

avg_t_plain = sum(times_no_rag) / len(times_no_rag)
avg_s_plain = sum(scores_no_rag) / len(scores_no_rag)

avg_t_rag = sum(times_rag) / len(times_rag)
avg_s_rag = sum(scores_rag) / len(scores_rag)
avg_node_s = sum(node_scores_rag) / len(node_scores_rag)


headers = ["METHOD", "AVG TIME (s)", "SEMANTIC SCORE", "NODE SCORE"]
rows = [
    ["PLAIN", avg_t_plain, avg_s_plain],
    ["WITH RAG", avg_t_rag, avg_s_rag, avg_node_s]
]

print(tabulate(rows, headers=headers, tablefmt="github", floatfmt=".4f"))
print(f"Setup Time: {setup_t:.4f}s")

| METHOD   |   AVG TIME (s) |   SEMANTIC SCORE |   NODE SCORE |
|----------|----------------|------------------|--------------|
| PLAIN    |         0.8233 |           0.7470 |              |
| WITH RAG |         8.9567 |           0.7501 |       0.7750 |
Setup Time: 6.7646s


**Advanced questions and case study**
We are listing here some additional questions for curious students. If you want to further challenge yourself give it a try!
In some case, there is no right answer, so feel free to explore different possibilities.

1. Research on methods to treat and cure cancer is an active research field. As such, information can get outdated really fast. Consider that you have documents containing both old and new information. How are you going to ensure that the new or more recent information is retrieved instead of older information?

2. AI systems are predictive, especially LLMs, which can hallucinate and ignore instructions, which can be really dangerous for medical application. For example, if someone is using your RAG pipeline to learn about cancer and then gets wrong treatment advice, it can be fatal for them. How can you restrict your generation process, so that it only provides information regarding cancer and does not start prescribing treatments and medications?

3. If you do not add a generation part to the pipeline, only have a retriever, does that differ too much from a search engine? If no, can you build a search engine with only embedding models and a retriever on a given set of documents?


*Note that these questions won't be considered in the final grading.*

In [ ]:
!pkill ollama